# Session 4 - CNN + Grad-CAM baseline


**GPU T4, ~2-3 h.** Trains a conventional classifier on the **DEV** split of the same
crops, explains it with Grad-CAM projected onto the region vocabulary, and audits it
on **TEST** with the identical FS/CR metric.

This provides a detector with high detection AUC and a saliency-based explanation
against which the language-based explanations can be compared.

In [ ]:
SESSION = "S4 CNN"

# ============================== CONFIG ==============================
ARCH             = "efficientnet_b0"   # or "legacy_xception"
EPOCHS           = 4
BATCH_SIZE       = 32
LR               = 3e-4
INPUT_SIZE       = 256
TRAIN_BUDGET_MIN = 120

AUDIT_SAMPLES    = 0      # 0 = all TEST samples (a CNN call is ~10 ms)
AUDIT_BUDGET_MIN = 300
SPLICE_FLOOR     = True
INPAINT          = "telea"
CKPT             = ""     # "" = train; or the path of an existing checkpoint

In [ ]:
# ---------------------------------------------------------------- BOOT
# Locates the code dataset wherever it is mounted, puts it on sys.path,
# prints the attached inputs, and records provenance.  The search is by
# file name, so the dataset's mount name does not matter.
import os, sys, subprocess, json, time

def _find_code():
    for root in ("/kaggle/input", "."):
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, files in os.walk(root):
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            if os.path.basename(dirpath) == "ccaudit" and "kaggle_utils.py" in files:
                return os.path.dirname(dirpath)
    raise FileNotFoundError(
        "Could not find the ccaudit package.\n"
        "Add Input -> your code dataset (<your-code-dataset>), and check that "
        "the preview shows ccaudit/kaggle_utils.py at the top level.")

CODE = _find_code()
if CODE not in sys.path:
    sys.path.insert(0, CODE)
# Child processes do not inherit sys.path.  Every `python -m ccaudit.<module>`
# below runs as a subprocess, so the code directory must be on PYTHONPATH.
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
from ccaudit import kaggle_utils as KU
from ccaudit import common as C

OUT = KU.work_dir("audit")
TMP = KU.temp_dir()
os.environ["HF_HOME"] = KU.temp_dir("hf")          # model weights stay out of /kaggle/working
os.environ["TOKENIZERS_PARALLELISM"] = "false"
KU.session_header(SESSION, OUT)
print("code:", CODE)

In [ ]:
# ------------------------------------------------------- SELF TEST (always)
# The self-test suite runs in under a minute and needs no dataset.  Each check
# corresponds to a failure mode that would produce plausible-looking but
# incorrect numbers, so a failure here invalidates everything that follows.
rc = KU.sh(f"{sys.executable} {CODE}/scripts/selftest.py", check=False)
if rc != 0:
    raise SystemExit("SELF TEST FAILED -- inspect the failures above before proceeding.")

In [ ]:
KU.pip_install("timm")
KU.gpu_report()
INDEX = KU.find_parsed_index()
if not INDEX:
    raise SystemExit("Add Input -> `cca-s1-parsed`.")
recs, meta = C.load_index(INDEX)
VOCAB = meta.get("vocab", "face8")
print("INDEX =", INDEX, "| samples:", len(recs))

In [ ]:
# ==================== TRAIN ON DEV ====================
# Re-uses a checkpoint from an attached earlier run if one is present.
found = [p for p in KU.find_checkpoints("cnn_*.pt")]
CKPT = CKPT or (found[0] if found else "")
if CKPT:
    print("resuming from", CKPT)
KU.sh(f'{sys.executable} -m ccaudit.m9_train_cnn --index "{INDEX}" '
      f'--out "{OUT}/cnn" --arch {ARCH} --epochs {EPOCHS} '
      f'--batch-size {BATCH_SIZE} --lr {LR} --input-size {INPUT_SIZE} '
      f'--train-budget-min {TRAIN_BUDGET_MIN}'
      + (f' --resume "{CKPT}"' if CKPT else ""),
      check=False, log=f"{OUT}/logs/m9.log")
log = C.load_json(f"{OUT}/cnn/train_log.json", {})
print("best val AUC:", log.get("best_val_auc"))
if (log.get("best_val_auc") or 0) < 0.9:
    print("!! val AUC below 0.9. Add epochs or try ARCH='legacy_xception' "
          "before auditing: a faithfulness measurement on a weak detector is "
          "of limited value.")

In [ ]:
# ==================== AUDIT ON TEST ====================
ckpt = f"{OUT}/cnn/cnn_{C.safe_name(ARCH)}.pt"
floor = " --splice-floor" if SPLICE_FLOOR else ""
inp   = f" --inpaint {INPAINT}" if INPAINT else ""
lim   = f" --limit-samples {AUDIT_SAMPLES}" if AUDIT_SAMPLES else ""
KU.sh(f'{sys.executable} -m ccaudit.m5_runner --index "{INDEX}" '
      f'--detector "cnn:{ckpt}" --out "{OUT}/run_cnn" --tag main '
      f'--split test{lim} --device cuda --vocab {VOCAB} '
      f'--time-budget-min {AUDIT_BUDGET_MIN}{floor}{inp}',
      check=False, log=f"{OUT}/logs/cnn_audit.log")

In [ ]:
# ==================== INTERIM METRICS ====================
KU.sh(f'{sys.executable} -m ccaudit.m6_metrics --raw "{OUT}/run_cnn" '
      f'--out "{OUT}/metrics_cnn" --vocab {VOCAB}', check=False)
for r in C.load_json(f"{OUT}/metrics_cnn/metrics.json", {}).get("results", []):
    print(f"{r['detector']}: AUC={r.get('AUC'):.3f} FS={r.get('FS'):.4f} "
          f"CR-prior={r.get('CR_minus_prior'):.3f} "
          f"faithful={r.get('faithful')}")

In [ ]:
NEXT_STEP = """1. Output tab -> New Dataset -> `cca-s4-cnn`.
2. Attach it to Session 11; the CNN appears as `cnn-<arch>` in every table.
3. If this notebook is re-run with `cca-s4-cnn` attached, the checkpoint is
   picked up automatically and training resumes instead of restarting."""

# ----------------------------------------------------------- WRAP UP
KU.disk_report()
print(C.banner("NEXT STEP"))
print(NEXT_STEP)